In [2]:
import pandas as pd
import numpy as np

# Set seed to get identical random noise on every run
np.random.seed(42)

# =====================================================================
# 1. SETUP: Creating the mock laboratory CSV log file
# =====================================================================
csv_filename = "laser_atp_raw_log.csv"

raw_csv_content = """Channel_ID,Sensor_Type,Voltage_V,Current_mA
CH_01,Photodiode,5.01,15.2
CH_02,Laser_Diode,12.45,142.8
CH_03,Thermal_Sensor,3.28,8.1
CH_04,Laser_Diode,12.38,139.5
CH_05,Photodiode,4.98,14.8
"""

# Write the data to a local text file
with open(csv_filename, "w", encoding="utf-8") as f:
    f.write(raw_csv_content.strip())

print(f"[INFO] Source file '{csv_filename}' created.\n")


# =====================================================================
# 2. PANDAS: Loading file and calculating power (P = V * I / 1000)
# =====================================================================

# Load CSV and set Channel_ID as the row names
df = pd.read_csv(csv_filename, index_col='Channel_ID')

# Calculate power column in Watts (without loops)
df['Power_W'] = (df['Voltage_V'] * df['Current_mA']) / 1000.0


# =====================================================================
# 3. FILTER & EXPORT: Catching failures (>1.5W) and saving to disk
# =====================================================================

# Find channels where power is greater than 1.5 Watts
power_failure_mask = df['Power_W'] > 1.5
failed_atp_channels = df[power_failure_mask]

# Select only the relevant columns for the final report
clean_failed_report = failed_atp_channels.loc[:, ['Sensor_Type', 'Voltage_V', 'Current_mA', 'Power_W']]

# Save the final failure report as a new CSV file
output_report_file = "failed_hardware_power_report.csv"
clean_failed_report.to_csv(output_report_file, index=True)


# =====================================================================
# 4. PRINT: Display results on Jupyter screen
# =====================================================================
print("=== HARDWARE POWER COMPLIANCE SCAN ===")
print(f"Total Out-of-Spec Channels Found: {len(clean_failed_report)}")
print("--------------------------------------------------\n")

print("--- FAILED CHANNELS DATA TABLE ---")
if len(clean_failed_report) > 0:
    print(clean_failed_report)
else:
    print("All channels are compliant.")
print("\n==================================================")
print(f"[SUCCESS] Filtered report saved as '{output_report_file}'")


[INFO] Source file 'laser_atp_raw_log.csv' created.

=== HARDWARE POWER COMPLIANCE SCAN ===
Total Out-of-Spec Channels Found: 2
--------------------------------------------------

--- FAILED CHANNELS DATA TABLE ---
            Sensor_Type  Voltage_V  Current_mA  Power_W
Channel_ID                                             
CH_02       Laser_Diode      12.45       142.8  1.77786
CH_04       Laser_Diode      12.38       139.5  1.72701

[SUCCESS] Filtered report saved as 'failed_hardware_power_report.csv'
